# StageNet + LRP (Layer-wise Relevance Propagation)

This notebook demonstrates **LRP** with **StageNet** for mortality prediction on **MIMIC-III**.

**LRP** decomposes a prediction into per-feature relevance scores in a single backward pass, with no baseline required. Relevances satisfy the conservation property: they sum to f(x).

**Rules:** Epsilon (numerically stable, smoother) and AlphaBeta (sharper, highlights positive evidence).

## Setup

In [1]:
from pathlib import Path

import torch
import numpy as np

from pyhealth.datasets import (
    MIMIC3Dataset,
    get_dataloader,
    split_by_patient,
)
from pyhealth.interpret.methods import LayerwiseRelevancePropagation
from pyhealth.models import StageNet
from pyhealth.tasks import MortalityPredictionMIMIC3
from pyhealth.trainer import Trainer

## Load MIMIC-III Dataset (Mounted from Campus Cluster)

## Data Access

**Full MIMIC-IV dataset** requires PhysioNet credentialing.
See the [Getting MIMIC Access Guide](https://docs.google.com/document/d/1NHgXzSPINafSg8Cd_whdfSauFXgh-ZflZIw5lu6k2T0/edit?usp=sharing) for instructions.

**Free 100-patient demo** (no credentialing required):
```
wget -r -N -c -np --user <physionet_user> --ask-password \
    https://physionet.org/files/mimic-iv-demo/2.2/
```
Or download via the PhysioNet web UI at: https://physionet.org/content/mimic-iv-demo/2.2/

### Required folder layout

`MIMIC4_EHR_ROOT` must be the directory that contains both `hosp/` and `icu/` subdirectories:

```
<MIMIC4_EHR_ROOT>/
├── hosp/
│   ├── patients.csv.gz        ← patients table
│   ├── admissions.csv.gz      ← admissions table
│   ├── diagnoses_icd.csv.gz   ← diagnoses_icd table
│   ├── procedures_icd.csv.gz  ← procedures_icd table
│   ├── labevents.csv.gz       ← labevents table
│   └── d_labitems.csv.gz      ← joined automatically by labevents
└── icu/
    └── icustays.csv.gz        ← always loaded (default table)
```

Set `MIMIC4_EHR_ROOT` in the cell below, then run the preflight check.

In [2]:
from pathlib import Path

# ── USING MIMIC-III FROM CAMPUS CLUSTER ──────────────────────────────────
# Mounted via sshfs from campus-cluster:/projects/illinois/eng/cs/jimeng/mimic3
# To mount: sshfs campus-cluster:/projects/illinois/eng/cs/jimeng/mimic3 ~/mimic-data
# To unmount: fusermount -u ~/mimic-data
# ──────────────────────────────────────────────────────────────────────────
MIMIC3_ROOT = str(Path.home() / "mimic-data")

# Tables required for MIMIC-III mortality prediction
REQUIRED_TABLES = [
    "DIAGNOSES_ICD",
    "PROCEDURES_ICD",
    "PRESCRIPTIONS",
]

# Preflight: verify required files exist
root = Path(MIMIC3_ROOT)
table_files = {
    "DIAGNOSES_ICD":  root / "DIAGNOSES_ICD.csv",
    "PROCEDURES_ICD": root / "PROCEDURES_ICD.csv",
    "PRESCRIPTIONS":  root / "PRESCRIPTIONS.csv",
    "ADMISSIONS":     root / "ADMISSIONS.csv",
    "PATIENTS":       root / "PATIENTS.csv",
}
missing = [str(p) for p in table_files.values() if not p.exists()]
if missing:
    print("⚠️  Missing files — check if mimic-data is mounted:")
    for f in missing:
        print(f"   {f}")
    print("\nTo mount: sshfs campus-cluster:/projects/illinois/eng/cs/jimeng/mimic3 ~/mimic-data")
else:
    print(f"✓ All required files found under: {MIMIC3_ROOT}")

⚠️  Missing files — check if mimic-data is mounted:
   /home/nemine/mimic-data/DIAGNOSES_ICD.csv
   /home/nemine/mimic-data/PROCEDURES_ICD.csv
   /home/nemine/mimic-data/PRESCRIPTIONS.csv
   /home/nemine/mimic-data/ADMISSIONS.csv
   /home/nemine/mimic-data/PATIENTS.csv

To mount: sshfs campus-cluster:/projects/illinois/eng/cs/jimeng/mimic3 ~/mimic-data


In [3]:
from pyhealth.datasets import MIMIC3Dataset

base_dataset = MIMIC3Dataset(
    root=MIMIC3_ROOT,
    tables=REQUIRED_TABLES,
    dev=True,  # Use dev mode for faster testing (1000 patients)
)

print(f"Loaded {len(base_dataset.unique_patient_ids)} patients")

No config path provided, using default config
Initializing mimic3 dataset from /home/nemine/mimic-data (dev mode: True)
No cache_dir provided. Using default cache dir: /home/nemine/.cache/pyhealth/dd96d4ce-639a-5750-851a-19cbdba4da7c
Found cached event dataframe: /home/nemine/.cache/pyhealth/dd96d4ce-639a-5750-851a-19cbdba4da7c/global_event_df.parquet
Found 1000 unique patient IDs
Loaded 1000 patients


## Apply Task and Create Sample Dataset

In [4]:
cache_dir = Path("../../mimic3_stagenet_lrp_cache_nb")

print("Applying mortality prediction task...")
sample_dataset = base_dataset.set_task(
    MortalityPredictionMIMIC3(),
)

print(f"Total samples: {len(sample_dataset)}")


Applying mortality prediction task...
Setting task MortalityPredictionMIMIC3 for mimic3 base dataset...
Task cache paths: task_df=/home/nemine/.cache/pyhealth/dd96d4ce-639a-5750-851a-19cbdba4da7c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/task_df.ld, samples=/home/nemine/.cache/pyhealth/dd96d4ce-639a-5750-851a-19cbdba4da7c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Found cached processed samples at /home/nemine/.cache/pyhealth/dd96d4ce-639a-5750-851a-19cbdba4da7c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.
Total samples: 208


/opt/workspace/PyHealth-fitzpa15/pyhealth/datasets/base_dataset.py:966: UserWarning: A newer version of litdata is available (0.2.61). Please consider upgrading with `pip install -U litdata`. Not all functionalities of the platform can be guaranteed to work with the current version.
  return SampleDataset(


## Split Dataset and Create DataLoaders

In [5]:
train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset, [0.8, 0.1, 0.1]
)

train_loader = get_dataloader(train_dataset, batch_size=64, shuffle=True)
val_loader = get_dataloader(val_dataset, batch_size=64, shuffle=False)
test_loader = get_dataloader(test_dataset, batch_size=1, shuffle=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 171 | Val: 20 | Test: 17


## Initialize and Train StageNet Model

In [6]:
model = StageNet(
    dataset=sample_dataset,
    embedding_dim=128,
    chunk_size=128,
    levels=3,
    dropout=0.3,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 7,183,625


In [7]:
trainer = Trainer(
    model=model,
    device="cpu",  # Change to "cuda" if GPU available
    metrics=["pr_auc", "roc_auc", "accuracy", "f1"],
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=5,
    monitor="roc_auc",
    optimizer_params={"lr": 1e-4},
)

StageNet(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(712, 128, padding_idx=0)
    (procedures): Embedding(258, 128, padding_idx=0)
    (drugs): Embedding(763, 128, padding_idx=0)
  ))
  (stagenet): ModuleDict(
    (conditions): StageNetLayer(
      (kernel): Linear(in_features=129, out_features=1542, bias=True)
      (recurrent_kernel): Linear(in_features=385, out_features=1542, bias=True)
      (nn_scale): Linear(in_features=384, out_features=64, bias=True)
      (nn_rescale): Linear(in_features=64, out_features=384, bias=True)
      (nn_conv): Conv1d(384, 384, kernel_size=(10,), stride=(1,))
      (relu): ReLU()
      (sigmoid): Sigmoid()
      (tanh): Tanh()
      (softmax): Softmax(dim=-1)
      (softmax_dim1): Softmax(dim=1)
      (nn_dropconnect): Dropout(p=0.3, inplace=False)
      (nn_dropconnect_r): Dropout(p=0.3, inplace=False)
      (nn_dropout): Dropout(p=0.3, inplace=False)
      (nn_dropres): Dropout(p=0.3, inplace=False)


Epoch 0 / 5:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'conditions' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'conditions' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'procedures' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'procedures' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.

--- Train epoch-0, step-3 ---
loss: 0.6978


Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'conditions' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'conditions' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'procedures' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'procedures' does not have mask information. Default mask will be created 

...

Epoch 4 / 5:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'conditions' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'conditions' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'procedures' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'procedures' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.

--- Train epoch-4, step-15 ---
loss: 0.6253


Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'conditions' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'conditions' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'procedures' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'procedures' does not have mask information. Default mask will be created 

--- Eval epoch-4, step-15 ---
pr_auc: 0.7667
roc_auc: 0.8627
accuracy: 0.8500
f1: 0.0000
loss: 0.6110
New best roc_auc score (0.8627) at epoch-4, step-15
Loaded best model


## Evaluate Model

In [8]:
results = trainer.evaluate(test_loader)
print("\nTest Results:")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")

Evaluation:   0%|          | 0/17 [00:00<?, ?it/s]/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'conditions' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'conditions' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'procedures' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'procedures' does not have mask information. Default mask will be created


Test Results:
  pr_auc: 0.2000
  roc_auc: 0.6000
  accuracy: 0.8235
  f1: 0.0000
  loss: 0.6241


## Get Test Sample for Interpretation

In [9]:
sample_batch = next(iter(test_loader))

# Get model prediction
with torch.no_grad():
    output = model(**sample_batch)
    probs = output["y_prob"]
    preds = torch.argmax(probs, dim=-1)
    true_label = sample_batch[model.label_key]

print("Model Prediction:")
print(f"  True label: {int(true_label[0].item())}")
print(f"  Predicted class: {int(preds[0].item())}")
print(f"  P(Survived): {probs[0, 0].item():.4f}")
if probs.shape[-1] > 1:
    print(f"  P(Died): {probs[0, 1].item():.4f}")

/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'conditions' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'conditions' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:436: UserWarning: Feature 'procedures' does not have time intervals. StageNet's temporal modeling capabilities will be limited. Consider using StageNet format with time intervals for better performance.
  warnings.warn(
/opt/workspace/PyHealth-fitzpa15/pyhealth/models/stagenet.py:452: UserWarning: Feature 'procedures' does not have mask information. Default mask will be created from embedded values. But it may not be accurate.

Model Prediction:
  True label: 0
  Predicted class: 0
  P(Survived): 0.4681


## Apply LRP with Epsilon Rule

In [10]:
# Initialize LRP with epsilon rule (numerically stable)
lrp_epsilon = LayerwiseRelevancePropagation(
    model, 
    rule="epsilon", 
    epsilon=0.01,
    use_embeddings=True
)

# Compute attributions
attributions_epsilon = lrp_epsilon.attribute(**sample_batch)

print("\nLRP Epsilon-Rule Attributions:")
for feature_key, attr in attributions_epsilon.items():
    total_relevance = attr[0].sum().item()
    positive = attr[0][attr[0] > 0].sum().item()
    negative = attr[0][attr[0] < 0].sum().item()
    
    print(f"\n{feature_key}:")
    print(f"  Shape: {attr.shape}")
    print(f"  Total relevance: {total_relevance:+.6f}")
    print(f"  Positive: {positive:+.6f} | Negative: {negative:+.6f}")


LRP Epsilon-Rule Attributions:

conditions:
  Shape: torch.Size([1, 17])
  Total relevance: +0.000179
  Positive: +0.000179 | Negative: +0.000000

procedures:
  Shape: torch.Size([1, 10])
  Total relevance: -0.000859
  Positive: +0.000000 | Negative: -0.000859

drugs:
  Shape: torch.Size([1, 174])
  Total relevance: -0.044270
  Positive: +0.000000 | Negative: -0.044270


## Apply LRP with AlphaBeta Rule

In [11]:
# Initialize LRP with alphabeta rule (sharper visualizations)
lrp_alphabeta = LayerwiseRelevancePropagation(
    model,
    rule="alphabeta",
    alpha=1.0,
    beta=0.0,
    use_embeddings=True
)

# Compute attributions
attributions_alphabeta = lrp_alphabeta.attribute(**sample_batch)

print("\nLRP AlphaBeta-Rule Attributions:")
for feature_key, attr in attributions_alphabeta.items():
    total_relevance = attr[0].sum().item()
    positive = attr[0][attr[0] > 0].sum().item()
    negative = attr[0][attr[0] < 0].sum().item()
    
    print(f"\n{feature_key}:")
    print(f"  Shape: {attr.shape}")
    print(f"  Total relevance: {total_relevance:+.6f}")
    print(f"  Positive: {positive:+.6f} | Negative: {negative:+.6f}")


LRP AlphaBeta-Rule Attributions:

conditions:
  Shape: torch.Size([1, 17])
  Total relevance: -0.932401
  Positive: +0.000000 | Negative: -0.932401

procedures:
  Shape: torch.Size([1, 10])
  Total relevance: -6.899529
  Positive: +0.000000 | Negative: -6.899529

drugs:
  Shape: torch.Size([1, 174])
  Total relevance: -8.034446
  Positive: +0.000000 | Negative: -8.034446


## Visualize Top Contributing Features

In [12]:
from pyhealth.medcode import InnerMap

# Lazy-loaded code description maps
_code_maps = {}


def _get_description(feature_key: str, code: str) -> str:
    """Look up a human-readable description for a clinical code."""
    if not code or code.startswith("<"):
        return "(special token)"
    vocab_map = {"conditions": "ICD9CM", "procedures": "ICD9PROC"}
    vocab = vocab_map.get(feature_key)
    if vocab is None:
        return code  # drugs: the string IS the description
    if vocab not in _code_maps:
        try:
            _code_maps[vocab] = InnerMap.load(vocab)
        except Exception:
            _code_maps[vocab] = None
    imap = _code_maps[vocab]
    if imap is None:
        return code
    try:
        desc = imap.lookup(code)
        return desc if desc else code
    except Exception:
        return code


def compute_token_grad_x_input(model, sample_batch, target_class_idx=1):
    """Per-token attribution via gradient × input.

    LRP through an implicit-loop RNN (StageNet calls the same nn.Linear in a
    Python for-loop, so forward hooks only retain the *last* timestep).
    PyTorch autograd correctly unrolls the full computation graph and produces
    distinct importance scores for every token position.

    Args:
        model: trained StageNet model.
        sample_batch: single-sample batch from the dataloader.
        target_class_idx: class to attribute toward (1 = mortality).

    Returns:
        Dict[str, Tensor[seq_len]] — gradient × input score per token.
    """
    model.eval()
    embedding_model = model.get_embedding_model()
    has_proc = hasattr(model, "dataset") and hasattr(
        getattr(model, "dataset", None), "input_processors"
    )

    emb_tensors, forward_kwargs = {}, {}

    for key in model.feature_keys:
        if key not in sample_batch:
            continue

        raw = sample_batch[key]
        if isinstance(raw, torch.Tensor):
            tok_tensor = raw                              # [1, seq_len]
        elif isinstance(raw, tuple):
            if has_proc and key in model.dataset.input_processors:
                schema = model.dataset.input_processors[key].schema()
                tok_tensor = raw[schema.index("value")]
            else:
                tok_tensor = raw[0]
        else:
            continue

        with torch.no_grad():
            emb = embedding_model({key: tok_tensor})[key]
        if emb.dim() == 4:
            emb = emb.sum(dim=2)
        emb = emb.detach().requires_grad_(True)
        emb_tensors[key] = emb

        # Non-padding mask [1, seq_len]
        if has_proc and key in model.dataset.input_processors:
            proc = model.dataset.input_processors[key]
            mask = (tok_tensor != 0).int() if proc.is_token() else (tok_tensor.abs().sum(-1) != 0).int()
        else:
            mask = (tok_tensor != 0).int()

        # forward_from_embedding receives (emb, mask); mask is picked up as the
        # optional trailing element when len(feature) == len(schema) + 1.
        forward_kwargs[key] = (emb, mask)

    for lk in getattr(model, "label_keys", []):
        if lk in sample_batch:
            forward_kwargs[lk] = sample_batch[lk]

    with torch.enable_grad():
        output = model.forward_from_embedding(**forward_kwargs)
        logits = output["logit"]          # [1, num_classes]
        if target_class_idx is None:
            target_class_idx = int(torch.argmax(logits[0]).item())
        score = logits[0, target_class_idx] if logits.shape[-1] > 1 else logits[0, 0]
        score.backward()

    result = {}
    for key, emb in emb_tensors.items():
        if emb.grad is not None:
            result[key] = (emb.grad * emb.detach()).sum(dim=-1)[0]    # [seq_len]
    return result


def show_top_features(lrp_attribs, gxi_attribs, feature_key,
                      sample_batch, model, top_k=10):
    """Show top tokens ranked by gradient×input; LRP feature relevance in header.

    Note: LRP through StageNet yields equal per-timestep relevance (see above),
    so LRP is reported as a feature-level total while gradient×input is used for
    within-feature token ranking.
    """
    if feature_key not in lrp_attribs:
        print(f"\nFeature '{feature_key}' not found. Available: {list(lrp_attribs.keys())}")
        return

    lrp_total = lrp_attribs[feature_key][0].sum().item()
    rank_attr = gxi_attribs.get(feature_key, lrp_attribs[feature_key][0]).flatten()

    # Token index → raw code string
    processor = model.dataset.input_processors.get(feature_key)
    idx_to_code = (
        {v: k for k, v in processor.code_vocab.items()}
        if processor is not None and hasattr(processor, "code_vocab")
        else {}
    )

    # Raw token indices from the batch
    raw = sample_batch.get(feature_key)
    if isinstance(raw, tuple):
        tok_tensor = raw[1][0]
    elif isinstance(raw, torch.Tensor):
        tok_tensor = raw[0]
    else:
        tok_tensor = None

    # Zero out padding before ranking so pad tokens never appear in top-k
    rank_masked = rank_attr.clone()
    if tok_tensor is not None:
        n = min(len(rank_masked), len(tok_tensor))
        rank_masked[:n][tok_tensor[:n] == 0] = 0.0

    k = min(top_k, int((rank_masked != 0).sum().item()))
    if k == 0:
        print(f"\nNo valid (non-padding) tokens found for '{feature_key}'")
        return

    _, top_pos = torch.topk(rank_masked.abs(), k=k)

    print(f"\nTop {k} tokens for '{feature_key}'  "
          f"(LRP feature relevance: {lrp_total:+.6f}  |  token ranking: gradient×input)")
    print(f"  {'#':<4} {'Grad×Input':>12}  {'Code':<22}  Description")
    print(f"  {'-'*4} {'-'*12}  {'-'*22}  {'-'*48}")

    for rank, pos in enumerate(top_pos.tolist(), 1):
        score = rank_masked[pos].item()

        if tok_tensor is not None and pos < len(tok_tensor):
            tok_int = int(tok_tensor[pos].item())
            code_str = idx_to_code.get(tok_int, f"<unk:{tok_int}>")
        else:
            code_str = f"<pos:{pos}>"

        desc = _get_description(feature_key, code_str)
        if len(desc) > 58:
            desc = desc[:55] + "..."

        print(f"  {rank:<4} {score:>+12.6f}  {code_str:<22}  {desc}")


# ── Gradient×input for per-token ranking (target: class 1 = died) ────────────
print("Computing gradient×input attributions for per-token ranking...")
gxi_attribs = compute_token_grad_x_input(model, sample_batch, target_class_idx=1)

print("\nAvailable features:", list(attributions_epsilon.keys()))

print("\n" + "="*80)
print("EPSILON RULE — Feature Relevance (LRP) + Token Ranking (Gradient×Input)")
print("="*80)
for feature in attributions_epsilon.keys():
    show_top_features(attributions_epsilon, gxi_attribs, feature, sample_batch, model, top_k=10)

print("\n" + "="*80)
print("ALPHABETA RULE — Feature Relevance (LRP) + Token Ranking (Gradient×Input)")
print("="*80)
for feature in attributions_alphabeta.keys():
    show_top_features(attributions_alphabeta, gxi_attribs, feature, sample_batch, model, top_k=10)


Computing gradient×input attributions for per-token ranking...

Available features: ['conditions', 'procedures', 'drugs']

EPSILON RULE — Feature Relevance (LRP) + Token Ranking (Gradient×Input)

Top 10 tokens for 'conditions'  (LRP feature relevance: +0.000179  |  token ranking: gradient×input)
  #      Grad×Input  Code                    Description
  ---- ------------  ----------------------  ------------------------------------------------
  1       -0.016504  99859                   Other postoperative infection
  2       -0.008901  0388                    Other specified septicemias
  3       -0.005627  5849                    Acute kidney failure, unspecified
  4       +0.002521  5672                    Other suppurative peritonitis
  5       -0.001787  56211                   Diverticulitis of colon (without mention of hemorrhage)
  6       +0.001122  1125                    Disseminated candidiasis
  7       -0.000952  E9308                   Other specified antibiotics causin

## Summary

- **Epsilon rule** (ε=0.01): Numerically stable, smoother attributions
- **AlphaBeta rule** (α=1.0, β=0.0): Sharper, highlights positive evidence
- **Conservation**: Σ(relevances) ≈ f(x)
- **Next steps**: Try different ε values, compare with IG/DeepLift/SHAP